# **BPS Statistics GIS Bridging Code**

Origin URL: https://sig.bps.go.id/bridging-kode/index

In [ ]:
import requests
import pandas as pd

api_url = 'https://sig.bps.go.id/rest-bridging/getwilayah'
period  = '2024_1.2022' #setup period in this section --> some options : "2019_1.2019", "2020_1.2020", "2023_1.2022", "2024_1.2022"

**Provinces** (provinsi)

In [ ]:
params = {'level': 'provinsi', 
          'parent': '0',
          'periode_merge': period}
response = requests.get(api_url, params=params)
provinces = pd.DataFrame(response.json())
print('Found',provinces.shape[0])
provinces.tail(10)

**Regencies / Cities** (kabupaten / kota)

In [ ]:
bps_province_kode = '31' #change in this section

params = {'level': 'kabupaten', 
          'parent': bps_province_kode,
          'periode_merge': period}
response = requests.get(api_url, params=params)
cities = pd.DataFrame(response.json())
print('Found',cities.shape[0])
cities.head(10)

**Districts** (kecamatan)

In [ ]:
bps_kabkot_kode = '3171' #change in this section

params = {'level': 'kecamatan', 
          'parent': bps_kabkot_kode,
          'periode_merge': period}
response = requests.get(api_url, params=params)
districts = pd.DataFrame(response.json())
print('Found',districts.shape[0])
districts.head(10)

**Sub-district / Village** (kelurahan / desa)

In [ ]:
bps_districts_kode = '3171010' #change in this section

params = {'level': 'desa', 
          'parent': bps_districts_kode,
          'periode_merge': period}
response = requests.get(api_url, params=params)
village = pd.DataFrame(response.json())
print('Found',village.shape[0])
village.head(10)

**All Villages in a Province** (semua desa dalam provinsi)

In [ ]:
bps_province_kode = '32' # Jawa Barat province code

# Step 1: Get all regencies/cities in the province
print(f"Fetching regencies/cities in province {bps_province_kode}...")
params = {'level': 'kabupaten', 
          'parent': bps_province_kode,
          'periode_merge': period}
response = requests.get(api_url, params=params)
all_regencies = pd.DataFrame(response.json())
print(f'Found {all_regencies.shape[0]} regencies/cities\n')

# Step 2: Get all districts in each regency
all_districts = []
for idx, regency in all_regencies.iterrows():
    regency_code = regency['kode_bps']
    regency_name = regency['nama_bps']
    print(f"Fetching districts in {regency_name} ({regency_code})...")
    
    params = {'level': 'kecamatan', 
              'parent': regency_code,
              'periode_merge': period}
    response = requests.get(api_url, params=params)
    districts = pd.DataFrame(response.json())
    all_districts.append(districts)
    print(f'  Found {districts.shape[0]} districts')

all_districts_df = pd.concat(all_districts, ignore_index=True)
print(f'\nTotal districts: {all_districts_df.shape[0]}\n')

# Step 3: Get all villages in each district
all_villages = []
for idx, district in all_districts_df.iterrows():
    district_code = district['kode_bps']
    district_name = district['nama_bps']
    print(f"Fetching villages in {district_name} ({district_code})...")
    
    params = {'level': 'desa', 
              'parent': district_code,
              'periode_merge': period}
    response = requests.get(api_url, params=params)
    villages = pd.DataFrame(response.json())
    all_villages.append(villages)
    print(f'  Found {villages.shape[0]} villages')

all_villages_df = pd.concat(all_villages, ignore_index=True)
print(f'\n=== TOTAL VILLAGES IN JAWA BARAT: {all_villages_df.shape[0]} ===')
all_villages_df.head(20)

In [ ]:
# Save to CSV
output_filename = f'all_villages_jawa_barat_{period}.csv'
all_villages_df.to_csv(output_filename, index=False)
print(f'✓ Successfully saved {all_villages_df.shape[0]} villages to: {output_filename}')

### **© Contributors**

- **Wahyu Calvin Frans Mariel** | *BPS - Statistics Indonesia* | [LinkedIn](https://www.linkedin.com/in/wahyu-calvin/)
